In [1]:
import itertools
import math

import cirq
import cirq_google
import numpy as np

# Google Quantum AI Challenge Problem

Your code may import from cirq, cirq_google, their dependencies, and python's standard library. Other imports are not allowed.

**Please upload your solutions to [this form](https://docs.google.com/forms/d/e/1FAIpQLScw93lwRb3OihfaHgpobs8TO6nXijwvkRVYjWPpNg7n92FjIA/viewform?usp=publish-editor).** See the form for further instructions.

Consider an $N$-qubit, weight $n$ Dicke state, defined by
$$|D^{(N)}_n\rangle = \begin{pmatrix} N \\ n \end{pmatrix}^{-1/2} \sum_k \mathcal{P}_k \left( |0\rangle^{\otimes(N-n)} \otimes |1\rangle^{\otimes n}\right),$$
where the sum is over all distinct permutations $\mathcal{P}_k$. For $n=1$, this is the $W$ state, given by
$$|W\rangle = \frac{1}{\sqrt N} \left( |100\ldots 0 \rangle + |010\ldots 0 \rangle + \ldots + |00\ldots 01 \rangle \right). $$

Your task is to complete the following function

In [2]:
import heapq

import networkx as nx


def native_x(qubit: cirq.Qid):
    """Return Pauli X as a Willow-native PhasedXZGate."""
    return cirq.PhasedXZGate(
        x_exponent=1.0,
        z_exponent=0.0,
        axis_phase_exponent=0.0,
    )(qubit)


def native_h(qubit: cirq.Qid):
    """Return Hadamard as a Willow-native PhasedXZGate."""
    return cirq.PhasedXZGate(
        x_exponent=0.5,
        z_exponent=1.0,
        axis_phase_exponent=1.5,
    )(qubit)


def native_ry(qubit: cirq.Qid, theta: float):
    """Return Ry(theta) as a Willow-native PhasedXZGate."""
    return cirq.PhasedXZGate(
        x_exponent=theta / np.pi,
        z_exponent=0.0,
        axis_phase_exponent=0.5,
    )(qubit)


def native_cnot(control: cirq.Qid, target: cirq.Qid):
    """Yield CNOT as H(target), CZ(control, target), H(target)."""
    yield native_h(target)
    yield cirq.CZ(control, target)
    yield native_h(target)


def build_cry_core(
    control: cirq.Qid,
    target: cirq.Qid,
    theta: float,
):
    """Decompose controlled Ry(2*theta)."""
    yield native_ry(target, theta)
    yield native_cnot(control, target)
    yield native_ry(target, -theta)
    yield native_cnot(control, target)


def build_gate_i(
    control: cirq.Qid,
    target: cirq.Qid,
    theta: float,
):
    """Build gate (i) from Bärtschi and Eidenbenz."""
    yield native_cnot(control, target)
    yield build_cry_core(target, control, theta)
    yield native_cnot(control, target)


def build_ccry_core(
    control_1: cirq.Qid,
    control_2: cirq.Qid,
    target: cirq.Qid,
    theta: float,
):
    """Decompose controlled-controlled Ry(2*theta)."""
    yield native_ry(target, theta / 2)
    yield native_cnot(control_2, target)
    yield native_ry(target, -theta / 2)
    yield native_cnot(control_1, target)
    yield native_ry(target, theta / 2)
    yield native_cnot(control_2, target)
    yield native_ry(target, -theta / 2)
    yield native_cnot(control_1, target)


def build_gate_ii(
    top: cirq.Qid,
    middle: cirq.Qid,
    bottom: cirq.Qid,
    theta: float,
):
    """Build gate (ii)_l from Bärtschi and Eidenbenz Figure 3."""
    yield native_cnot(top, bottom)
    yield build_ccry_core(middle, bottom, top, theta)
    yield native_cnot(top, bottom)


def build_scs_block(
    path_qubits: list[cirq.Qid],
    m: int,
    k: int,
):
    """Construct SCS_{m,k} on path_qubits[:m]."""
    sub_qubits = path_qubits[:m]
    theta_i = np.arccos(1.0 / np.sqrt(m))
    yield build_gate_i(
        sub_qubits[m - 2],
        sub_qubits[m - 1],
        theta_i,
    )

    for excitation in range(2, min(k, m - 1) + 1):
        top = sub_qubits[m - excitation - 1]
        middle = sub_qubits[m - excitation]
        bottom = sub_qubits[m - 1]
        theta_ii = np.arccos(np.sqrt(excitation / m))
        yield build_gate_ii(top, middle, bottom, theta_ii)


def build_unrouted_u_nk(
    path_qubits: list[cirq.Qid],
    num_qubits: int,
    num_excitations: int,
) -> cirq.Circuit:
    """Construct U_{N,n} before enforcing hardware connectivity."""
    circuit = cirq.Circuit()
    for qubit in path_qubits[num_qubits - num_excitations :]:
        circuit.append(native_x(qubit))

    for size in range(num_qubits, num_excitations, -1):
        circuit.append(
            build_scs_block(
                path_qubits,
                m=size,
                k=num_excitations,
            )
        )

    for size in range(num_excitations, 1, -1):
        circuit.append(
            build_scs_block(
                path_qubits,
                m=size,
                k=size - 1,
            )
        )

    return circuit


def build_full_u_nk(
    path_qubits: list[cirq.Qid],
    num_qubits: int,
    num_excitations: int,
) -> cirq.Circuit:
    """Construct and route U_{N,n} over the selected active path."""
    unrouted = build_unrouted_u_nk(
        path_qubits,
        num_qubits,
        num_excitations,
    )
    path_graph = nx.Graph()
    path_graph.add_nodes_from(path_qubits)
    path_graph.add_edges_from(zip(path_qubits, path_qubits[1:]))
    routed = cirq.RouteCQC(path_graph)(
        unrouted,
        initial_mapper=cirq.HardCodedInitialMapper(
            {qubit: qubit for qubit in path_qubits}
        ),
    )
    return cirq.optimize_for_target_gateset(
        routed,
        gateset=cirq.CZTargetGateset(allow_partial_czs=False),
    )


class MultiModePathfinder:
    """Find a low-error connected qubit chain on Willow."""

    def __init__(self, mode: str = "branch_and_bound"):
        self.mode = mode

    def calculate_cz_counts(
        self,
        num_qubits: int,
        num_excitations: int,
    ) -> dict[int, int]:
        """Count emitted CZ gates on each relative path coupler."""
        counts = {index: 0 for index in range(num_qubits - 1)}
        logical_qubits = list(cirq.LineQubit.range(num_qubits))
        positions = {
            qubit: index for index, qubit in enumerate(logical_qubits)
        }
        circuit = build_full_u_nk(
            logical_qubits,
            num_qubits,
            num_excitations,
        )
        for operation in circuit.all_operations():
            if operation.gate != cirq.CZ:
                continue
            first, second = (
                positions[qubit] for qubit in operation.qubits
            )
            if abs(first - second) != 1:
                raise ValueError("Routing left a non-nearest CZ")
            counts[min(first, second)] += 1
        return counts

    @staticmethod
    def _step_cost(
        start: cirq.Qid,
        destination: cirq.Qid,
        step_index: int,
        sq_errors: dict,
        cz_errors: dict,
        readout_errors: dict,
        cz_counts: dict[int, int],
    ) -> float:
        sq_error = sq_errors.get(destination, 0.001)
        readout_error = readout_errors.get(destination, 0.01)
        cz_error = cz_errors.get(
            (start, destination),
            cz_errors.get((destination, start), 0.01),
        )
        return (
            -np.log(max(1.0 - sq_error, 1e-6))
            - np.log(max(1.0 - readout_error, 1e-6))
            + cz_counts.get(step_index, 1)
            * -np.log(max(1.0 - cz_error, 1e-6))
        )

    def find_path(
        self,
        qubits: list[cirq.GridQubit],
        couplers: list[tuple[cirq.GridQubit, cirq.GridQubit]],
        num_qubits: int,
        num_excitations: int,
        sq_errors: dict,
        cz_errors: dict,
        readout_errors: dict,
    ):
        """Find the selected mode's minimum-cost simple path."""
        adjacency = {qubit: [] for qubit in qubits}
        for first, second in couplers:
            if first in adjacency and second in adjacency:
                adjacency[first].append(second)
                adjacency[second].append(first)

        cz_counts = self.calculate_cz_counts(
            num_qubits,
            num_excitations,
        )
        if self.mode == "dijkstra":
            return self._dijkstra(
                qubits,
                adjacency,
                num_qubits,
                sq_errors,
                cz_errors,
                readout_errors,
                cz_counts,
            )
        return self._branch_and_bound(
            qubits,
            adjacency,
            num_qubits,
            sq_errors,
            cz_errors,
            readout_errors,
            cz_counts,
        )

    def _branch_and_bound(
        self,
        qubits,
        adjacency,
        num_qubits,
        sq_errors,
        cz_errors,
        readout_errors,
        cz_counts,
    ):
        best_path = None
        best_cost = float("inf")

        def search(path, cost, visited):
            nonlocal best_path, best_cost
            if cost >= best_cost:
                return
            if len(path) == num_qubits:
                best_path = list(path)
                best_cost = cost
                return

            current = path[-1]
            step_index = len(path) - 1
            neighbors = [
                neighbor
                for neighbor in adjacency[current]
                if neighbor not in visited
            ]
            neighbors.sort(
                key=lambda neighbor: cz_errors.get(
                    (current, neighbor),
                    cz_errors.get((neighbor, current), 0.01),
                )
            )
            for neighbor in neighbors:
                step_cost = self._step_cost(
                    current,
                    neighbor,
                    step_index,
                    sq_errors,
                    cz_errors,
                    readout_errors,
                    cz_counts,
                )
                visited.add(neighbor)
                path.append(neighbor)
                search(path, cost + step_cost, visited)
                path.pop()
                visited.remove(neighbor)

        for start in qubits:
            sq_error = sq_errors.get(start, 0.001)
            readout_error = readout_errors.get(start, 0.01)
            initial_cost = (
                -np.log(max(1.0 - sq_error, 1e-6))
                - np.log(max(1.0 - readout_error, 1e-6))
            )
            search([start], initial_cost, {start})

        return best_path, best_cost

    def _dijkstra(
        self,
        qubits,
        adjacency,
        num_qubits,
        sq_errors,
        cz_errors,
        readout_errors,
        cz_counts,
    ):
        queue = []
        counter = 0
        for start in qubits:
            sq_error = sq_errors.get(start, 0.001)
            readout_error = readout_errors.get(start, 0.01)
            initial_cost = (
                -np.log(max(1.0 - sq_error, 1e-6))
                - np.log(max(1.0 - readout_error, 1e-6))
            )
            heapq.heappush(
                queue,
                (initial_cost, counter, (start,), {start}),
            )
            counter += 1

        while queue:
            cost, _, path, visited = heapq.heappop(queue)
            if len(path) == num_qubits:
                return list(path), cost

            current = path[-1]
            step_index = len(path) - 1
            for neighbor in adjacency[current]:
                if neighbor in visited:
                    continue
                step_cost = self._step_cost(
                    current,
                    neighbor,
                    step_index,
                    sq_errors,
                    cz_errors,
                    readout_errors,
                    cz_counts,
                )
                next_visited = set(visited)
                next_visited.add(neighbor)
                heapq.heappush(
                    queue,
                    (
                        cost + step_cost,
                        counter,
                        path + (neighbor,),
                        next_visited,
                    ),
                )
                counter += 1

        return None, float("inf")


def create_dicke_state(
    num_qubits: int,
    num_excitations: int,
    cz_pauli_error_rate: dict[
        tuple[cirq.GridQubit, cirq.GridQubit], tuple[float, float]
    ],
    sq_pauli_error_rate: dict[cirq.GridQubit, tuple[float, float]],
    zero_errors: dict[cirq.GridQubit, tuple[float, float]],
    one_errors: dict[cirq.GridQubit, tuple[float, float]],
) -> tuple[cirq.Circuit, set[cirq.GridQubit]]:
    """Create a Willow-native circuit that prepares a Dicke state."""
    cz_errors = {
        pair: error
        for pair, (error, _) in cz_pauli_error_rate.items()
    }
    sq_errors = {
        qubit: error
        for qubit, (error, _) in sq_pauli_error_rate.items()
    }
    readout_errors = {
        qubit: (
            zero_errors.get(qubit, (0.01, 0))[0]
            + one_errors.get(qubit, (0.01, 0))[0]
        )
        / 2.0
        for qubit in sq_pauli_error_rate
    }

    pathfinder = MultiModePathfinder(mode="branch_and_bound")
    best_path, _ = pathfinder.find_path(
        qubits=list(sq_pauli_error_rate),
        couplers=list(cz_pauli_error_rate),
        num_qubits=num_qubits,
        num_excitations=num_excitations,
        sq_errors=sq_errors,
        cz_errors=cz_errors,
        readout_errors=readout_errors,
    )
    if best_path is None:
        raise NotImplementedError(
            f"No active connected path for N={num_qubits}"
        )

    circuit = build_full_u_nk(
        best_path,
        num_qubits,
        num_excitations,
    )
    circuit = cirq.drop_negligible_operations(circuit, atol=1e-8)
    return circuit, set(best_path)


For small $N$, you may check that your circuit produces the correct state using the following function:

In [3]:
def create_dicke_statevector(num_qubits: int, num_excitations: int) -> np.ndarray:
    psi = np.zeros(2**num_qubits)
    normalization = 1 / np.sqrt(math.comb(num_qubits, num_excitations))
    qubits = range(num_qubits)
    for excited_qubits in itertools.combinations(qubits, num_excitations):
        idx = sum(2**qubit_number for qubit_number in excited_qubits)
        psi[idx] = normalization
    return psi


def test_circuit(circuit: cirq.Circuit, num_excitations: int, qubits: set[cirq.GridQubit], sampler = cirq.Simulator()):
    all_qubits = circuit.all_qubits()
    ancilla_qubits = all_qubits.difference(qubits)
    ordered_qubits = sorted(qubits) + sorted(ancilla_qubits)
    keep_indices = range(qubits)
    num_qubits = len(qubits)
    psi_circuit_full = sampler.simulate(circuit, qubit_order=ordered_qubits).final_state_vector
    psi_circuit = cirq.sub_state_vector(
        state_vector=psi_circuit_full,
        keep_indices=keep_indices
    ) if len(all_qubits) > len(qubits) else psi_circuit_full
    assert np.isclose(np.abs(psi_circuit @ create_dicke_statevector(num_qubits, num_excitations)) ** 2, 1.0)

### Requirements:
* The resulting circuit is allowed to contain only
  * CZ gates between pairs of qubits included in `cz_pauli_error_rate`, which will be a subset (and most) of the connected pairs in the [spec sheet](https://quantumai.google/static/site-assets/downloads/willow-spec-sheet.pdf).
  * Arbitrary single-qubit unitaries (`cirq.PhasedXZGate`) on qubits included in `sq_pauli_error_rate`, which will be most of the qubits in the spec sheet.
  * Optionally, mid-circuit reset gates.
* Classically-conditioned gates are not supported.
* In the absence of noise, the circuit must either produce perfect Dicke states of the requested $N$ and $n$ or throw a `NotImplementedError`.
* If your circuit construction code takes longer than a minute to return a circuit, that is counted the same as returning a `NotImplementedError`.
* Any circuits containing more than 1750 moments will not be run.

# Scoring

We prepare the Dicke state on actual quantum hardware and measure it in both the Z basis and the X basis. We use the following `overall_score` function to compare against the expected distributions.

In [4]:
def check_z_bitstring_distribution(
    bitstrings: np.ndarray, num_excitations: int
) -> float:
    """Measure how close the observed Z-basis distribution of bitstrings is to
    the ideal one. The lower the score, the better.

    Args:
        bitstrings: An array of shape (n_shots, n_qubits) containing the n_shots Z-basis bitstrings.
        num_excitations: The weight, n, of the Dicke state.

    Returns:
        A measure of how close the distribution is to the ideal one.
    """
    # first, let's see how many bitstrings have one excitation
    n_shots, n_qubits = bitstrings.shape
    measured_excitations = np.sum(bitstrings, axis=1)
    fraction_in_subspace = np.mean(measured_excitations == num_excitations)

    # now, let's check how uniform the distribution is over states with n excitations
    chi2 = 0
    ideal_prob = 1 / math.comb(n_qubits, num_excitations)
    for indices in itertools.combinations(range(n_qubits), num_excitations):
        ideal_state = np.zeros(n_qubits)
        ideal_state[np.array(indices)] = 1
        match_count = np.sum(np.all(bitstrings == ideal_state, axis=1))
        expected_count = n_shots * fraction_in_subspace * ideal_prob
        chi2 += (match_count - expected_count) ** 2 / expected_count

    return ideal_prob * chi2 / fraction_in_subspace


def get_correlator_average(bitstrings: np.ndarray, correlator_size: int) -> float:
    """Get the average of all weight-correlator_size correlators in `bitstrings`.

    Args:
        bitstrings: The input bitstrings, shape (n_shots, n_qubits).
        correlator_size: Compute all correlators of this weight.

    Returns:
        The average of all weight-k correlators.
    """
    correlator_sum = 0.0
    n_shots, n_qubits = bitstrings.shape
    for indices in itertools.combinations(range(n_qubits), correlator_size):
        correlator_sum += 1 - 2 * np.mean(np.sum(bitstrings[:, indices], axis=1) % 2)
    return correlator_sum / math.comb(n_qubits, correlator_size)


def check_x_bitstring_distribution(
    bitstrings: np.ndarray, num_excitations: int, max_correlator_size: int
) -> float:
    """Check expected correlations in the X basis. The lower the score, the better.

    Args:
        bitstrings: An array of shape (n_shots, n_qubits) containing the n_shots X-basis bitstrings.
        max_correlator_size: The maximum weight of correlators to consider.
        num_excitations: The weight, n, of the Dicke state.

    Returns:
        A measure of how the correlations in the X-basis measurements compare to expectation.
    """
    comb = lambda n, k: math.comb(n, k) if (k >= 0 and n >= k) else 0
    n_shots, n_qubits = bitstrings.shape
    score = 0.0
    for correlator_size in range(1, max_correlator_size + 1):
        expected_correlator = (
            comb(correlator_size, correlator_size // 2)
            * comb(n_qubits - correlator_size, num_excitations - correlator_size // 2)
            / comb(n_qubits, num_excitations)
            if correlator_size % 2 == 0
            else 0.0
        )
        measured_correlator = get_correlator_average(bitstrings, correlator_size)
        score += (
            (expected_correlator - measured_correlator) ** 2 / expected_correlator**2
            if expected_correlator != 0
            else (expected_correlator - measured_correlator) ** 2
        )

    return score


def overall_score(
    z_bitstrings: np.ndarray,
    x_bitstrings: np.ndarray,
    num_excitations: int,
    max_correlator_size: int,
) -> float:
    """An overall score that checks how the Z and X basis bitstrings compare to their ideal
    distributions. The higher the score, the better.

    Args:
        z_bitstrings: The bitstrings measured in the Z basis
        x_bitstrings: The bitstrings measured in the X basis
        num_excitations: The weight, n, of the Dicke state.
        max_correlator_size: The maximum weight of correlators to consider

    Returns:
        A measure of how similar the bitstrings are to their ideal distributions.
    """
    return 1 / (
        check_z_bitstring_distribution(z_bitstrings, num_excitations)
        + check_x_bitstring_distribution(
            x_bitstrings, num_excitations, max_correlator_size
        )
    )


### Judging criteria:
* We will run circuits generated by your code for various choices of `num_qubits` and `num_excitations` on real quantum hardware, measuring in the Z basis as well as in the X basis, each for 10,000 shots.
* For each choice of `num_qubits` and `num_excitations`, we use the `overall_score` function to score the performance. We then sum this over the choices of `num_qubits`.
* The solution with the highest score wins.

### Things to consider:
* It is permitted to use AI to answer these questions, but please include any prompts you gave to the AI in your submission.
* This will be run on real quantum hardware, so techniques like dynamical decoupling are important.


You may use the following functions to generate fake error rates to test your algorithm.

In [5]:
def pick_single_qubit_dropouts(
    dropout_rate: float,
    rng: np.random.Generator = np.random.default_rng()
) -> set[cirq.GridQubit]:
    """Pick some qubits to be broken.

    Args:
        dropout_rate: The probability that a qubit is broken.
        rng: The pseudo-random number generator

    Returns:
        The broken qubits.
    """
    all_qubits = np.array(list(cirq_google.Willow105.metadata.qubit_set))
    is_dropout = rng.random(len(all_qubits)) < dropout_rate
    return set(all_qubits[is_dropout])


def generate_qubit_error_rates(
    median: float,
    spread: float,
    dropouts: set[cirq.GridQubit],
    uncertainty: float,
    rng: np.random.Generator = np.random.default_rng(),
) -> dict[cirq.GridQubit, tuple[float, float]]:
    """Generate fake single-qubit error rates.

    Args:
        median: The median error rate.
        spread: The spread of the lognormal distribution
        dropouts: Qubits to exclude
        uncertainty: The uncertainty of the measured error rates.
        rng: The pseudo-random number generator

    Returns:
        A dictionary from qubits to the measured error rate and statistical uncertainty.
    """

    all_qubits = np.array(list(cirq_google.Willow105.metadata.qubit_set))
    is_working = np.array([q not in dropouts for q in all_qubits])
    working_qubits = all_qubits[is_working]
    error_rate = rng.lognormal(np.log(median), spread, len(working_qubits))
    return dict(
        zip(working_qubits, zip(error_rate, uncertainty * np.ones(len(working_qubits))))
    )


def generate_two_qubit_pauli_error_rates(
    median: float = 2.5e-3,
    spread: float = 0.5,
    dropout_rate: float = 0.01,
    uncertainty: float = 5e-4,
    rng: np.random.Generator = np.random.default_rng(),
    single_qubit_dropouts: set[cirq.GridQubit] = None,
) -> dict[tuple[cirq.GridQubit, cirq.GridQubit], tuple[float, float]]:
    """Generate fake two-qubit error rates.

    Args:
        median: The median error rate.
        spread: The spread of the lognormal distribution
        dropout_rate: The probability of a broken coupler.
        uncertainty: The uncertainty of the measured error rates.
        rng: The pseudo-random number generator
        single_qubit_dropouts: The single-qubit dropouts to skip.

    Returns:
        A dictionary from qubits to the measured error rate and statistical uncertainty.
    """
    if single_qubit_dropouts is None:
        single_qubit_dropouts = set()

    all_pairs = np.array(list(cirq_google.Willow105.metadata.qubit_pairs))
    no_qubit_dropouts = np.array(
        [np.all([q not in single_qubit_dropouts for q in pair]) for pair in all_pairs]
    )
    is_working = (rng.random(len(all_pairs)) > dropout_rate) * no_qubit_dropouts
    working_pairs = [tuple(pair) for pair in all_pairs[is_working]]
    error_rate = rng.lognormal(np.log(median), spread, len(working_pairs))
    return dict(
        zip(working_pairs, zip(error_rate, uncertainty * np.ones(len(working_pairs))))
    )


In [6]:
single_qubit_dropouts = pick_single_qubit_dropouts(0.01)
sq_pauli_error_rate = generate_qubit_error_rates(4e-4, 0.5, single_qubit_dropouts, 2e-4)
cz_pauli_error_rate = generate_two_qubit_pauli_error_rates(
    single_qubit_dropouts=single_qubit_dropouts
)
zero_errors = generate_qubit_error_rates(5e-3, 0.5, single_qubit_dropouts, 2e-4)
one_errors = generate_qubit_error_rates(1e-2, 0.5, single_qubit_dropouts, 3e-4)

# Bonus question:
Now suppose we allow for adaptive circuits, i.e. those with mid-circuit measurements and subsequent gates conditioned on the outcomes of those measurements. See if you can construct shallower circuits to prepare Dicke states given this capability. We will not run these circuits on the hardware.

For all $(N, n)$, with $2 \leq N \leq 15$, and $1 \leq n \leq \lfloor N/2 \rfloor$, we will compare the depth of the circuits produced by the various submissions and assign a point to the one with the shallowest depth, measured by the number of moments in the circuit.

Specifically, your task is to complete the following function:

In [8]:
def create_dicke_state_adaptive(
    num_qubits: int, num_excitations: int
) -> tuple[cirq.Circuit, set[cirq.GridQubit]]:
    """Create a Cirq circuit that produces a Dicke state, allowing for adaptive circuits.

    Args:
        num_qubits: The number, N, of qubits in the Dicke state.
        num_excitations: The number, n, of 1s in the state.

    Returns:
        A Cirq circuit to produce Dicke state and a set indicating which qubits the resulting state lives on.

    Raises:
        NotImplementedError: The requested (N, n) combination is not implemented.
    """

## Scoring

We will use the following function to score submissions:

In [7]:
def score_solution() -> float:
    score = 0
    for num_qubits in range(2, 21):
        for num_excitations in range(1, num_qubits//2+1):
            try:
                circuit, qubits = create_dicke_state_adaptive(num_qubits, num_excitations)
            except NotImplementedError:
                continue
            validate_circuit(circuit, num_excitations, qubits) # checks that the circuit is valid
            score += num_qubits * num_excitations / len(circuit)
    return score

Notes and constraints:
* We will verify classically that the circuits produce the desired Dicke state up to a global phase.
* Using adaptive circuits is not required, so you may reuse your solution from the previous question.
* Although we do not provide error rates for this question, it is still necessary to respect the connectivity of the [Willow device](https://quantumai.google/static/site-assets/downloads/willow-spec-sheet.pdf), i.e. you can only use qubits on that device and CZ gates between adjacent qubits on the device. You may use `cirq_google.Willow105.validate_circuit(circuit)` to validate that your circuits are correctly routed to such a device.
* On real hardware, qubits along a diagonal such that row # + column # = const share a readout line and measuring one qubit dephases the other qubits on that readout line. Therefore, mid-circuit measurements of some but not all of the qubits on a readout line are not allowed. (This constraint is not checked by `cirq_google.Willow105.validate_circuit(circuit)`.)
* As before, the code must execute in under a minute for each $(N, n)$.

In [9]:
# Run the complete standard-challenge validation matrix (56 cases).
# This verifies the noiseless result and submission contract locally; the final
# noisy competition score can only be measured on Google's quantum hardware.
import time


def _submission_fidelity(
    circuit: cirq.Circuit,
    result_qubits: set[cirq.GridQubit],
    num_excitations: int,
) -> float:
    ordered_qubits = sorted(result_qubits)
    state = cirq.Simulator(dtype=np.complex128).simulate(
        circuit,
        qubit_order=ordered_qubits,
    ).final_state_vector
    ideal = create_dicke_statevector(len(ordered_qubits), num_excitations)
    return float(np.abs(state @ ideal) ** 2)


def _contract_violations(
    circuit: cirq.Circuit,
    result_qubits: set[cirq.GridQubit],
    num_qubits: int,
    construction_seconds: float,
) -> list[str]:
    violations = []
    active_qubits = set(sq_pauli_error_rate)
    active_couplers = {frozenset(pair) for pair in cz_pauli_error_rate}

    if len(result_qubits) != num_qubits:
        violations.append(f"returned {len(result_qubits)} qubits, expected {num_qubits}")
    if not result_qubits.issubset(active_qubits):
        violations.append("uses unavailable result qubits")
    if len(circuit) > 1750:
        violations.append(f"depth {len(circuit)} exceeds 1750 moments")
    if construction_seconds >= 60:
        violations.append(f"construction took {construction_seconds:.3f}s")

    for operation in circuit.all_operations():
        gate = operation.gate
        if isinstance(gate, cirq.PhasedXZGate):
            if operation.qubits[0] not in active_qubits:
                violations.append(f"single-qubit gate uses unavailable {operation.qubits[0]}")
        elif gate == cirq.CZ:
            if frozenset(operation.qubits) not in active_couplers:
                violations.append(f"CZ uses unavailable coupler {operation.qubits}")
        elif isinstance(gate, cirq.ResetChannel):
            if any(qubit not in active_qubits for qubit in operation.qubits):
                violations.append(f"reset uses unavailable qubits {operation.qubits}")
        else:
            violations.append(f"unsupported operation {operation!r}")

    try:
        cirq_google.Willow105.validate_circuit(circuit)
    except ValueError as error:
        violations.append(f"Willow validation failed: {error}")

    return list(dict.fromkeys(violations))


results = []
print("DuQuantum full standard-challenge report")
print("=" * 96)
for num_qubits in range(2, 16):
    for num_excitations in range(1, num_qubits // 2 + 1):
        started = time.perf_counter()
        circuit, result_qubits = create_dicke_state(
            num_qubits,
            num_excitations,
            cz_pauli_error_rate,
            sq_pauli_error_rate,
            zero_errors,
            one_errors,
        )
        elapsed = time.perf_counter() - started
        fidelity = _submission_fidelity(circuit, result_qubits, num_excitations)
        violations = _contract_violations(
            circuit,
            result_qubits,
            num_qubits,
            elapsed,
        )
        passed = bool(
            np.isclose(fidelity, 1.0, rtol=0.0, atol=1e-8)
            and not violations
        )
        results.append((passed, fidelity, len(circuit), elapsed))
        print(
            f"N={num_qubits:2d} n={num_excitations:2d} "
            f"fidelity={fidelity:.12f} moments={len(circuit):4d} "
            f"build={elapsed:7.3f}s {'PASS' if passed else 'FAIL'}"
        )
        for violation in violations:
            print(f"    - {violation}")

passed_count = sum(result[0] for result in results)
print("=" * 96)
print(f"Cases passed: {passed_count}/{len(results)}")
print(f"Minimum fidelity: {min(result[1] for result in results):.12f}")
print(f"Maximum circuit depth: {max(result[2] for result in results)}/1750 moments")
print(f"Maximum construction time: {max(result[3] for result in results):.3f}/60 seconds")

DuQuantum full standard-challenge report
N= 2 n= 1 fidelity=1.000000000000 moments=   5 build=  1.594s PASS
N= 3 n= 1 fidelity=1.000000000000 moments=   9 build=  0.039s PASS
N= 4 n= 1 fidelity=1.000000000000 moments=  13 build=  0.069s PASS
N= 4 n= 2 fidelity=1.000000000000 moments=  43 build=  0.152s PASS
N= 5 n= 1 fidelity=1.000000000000 moments=  17 build=  0.059s PASS
N= 5 n= 2 fidelity=1.000000000000 moments=  71 build=  0.153s PASS
N= 6 n= 1 fidelity=1.000000000000 moments=  21 build=  0.081s PASS
N= 6 n= 2 fidelity=1.000000000000 moments=  93 build=  0.161s PASS
N= 6 n= 3 fidelity=1.000000000000 moments= 159 build=  0.292s PASS
N= 7 n= 1 fidelity=1.000000000000 moments=  25 build=  0.107s PASS
N= 7 n= 2 fidelity=1.000000000000 moments= 127 build=  0.281s PASS
N= 7 n= 3 fidelity=1.000000000000 moments= 199 build=  0.387s PASS
N= 8 n= 1 fidelity=1.000000000000 moments=  29 build=  0.118s PASS
N= 8 n= 2 fidelity=1.000000000000 moments= 155 build=  3.360s PASS
N= 8 n= 3 fidelity=1.